# RTMA hourly EMC export

Pulls NWS Real-Time Mesoscale Analysis (`NOAA/NWS/RTMA`, hourly 2.5 km, 2011–present),
computes **per-pixel** RH, VPD, and EMC (so the nonlinear EMC(T, RH) is not biased by
`EMC(mean(T), mean(RH))`), then reduces to pyrome-mean hourly scalars and exports CSVs.

The downstream `fb_tools.weather.rtma` module consumes these CSVs to produce
FlamMap percentile-scenario FM (FM1/FM10/FM100 via hourly NFDRS78 time-lag with
Bradshaw 1984 precip saturation-stall; FM_herb/FM_woody via GSI).

**Schema** (per export task — one per year, all 9 CO pyromes):

```
pyrome_id, datetime_utc, tmp_f, rh_pct, emc_pct, vpd_pa, pcp_mm_hr
```

ERC stays GridMET-derived (FSPro/FSim daily contract). HRRR stays the wind source.
This pipeline only upgrades the dead-FM and (optionally) live-FM legs of the
per-pyrome FlamMap scenarios.

In [ ]:
import ee

ee.Authenticate()
ee.Initialize(project='cfri-ee')
print('GEE authenticated.')

## Pyrome geometries (CO analysis extent)

The CO pyromes per `CLAUDE.md`: `42, 43, 45, 46, 47, 52, 53, 56, 128`.

In [ ]:
CO_PYROME_IDS = [42, 43, 45, 46, 47, 52, 53, 56, 128]

pyromes = ee.FeatureCollection('projects/cfri-ee/assets/weather/Pyromes_CONUS_20200206')
co_pyromes = pyromes.filter(ee.Filter.inList('PYROME', CO_PYROME_IDS))
print('CO pyromes:', co_pyromes.size().getInfo())
co_pyromes.aggregate_array('PYROME').getInfo()

## Fire-environment mask (MODIS burned area)

Restricts all subsequent RTMA spatial reductions to historically burned pixels.
Created once here; referenced as `fire_mask` inside `reduce_to_pyromes()`.

In [ ]:
# ── Fire-environment mask from MODIS burned area (2001–2022) ──────────────
# Restricts RTMA spatial averaging to pixels that have historically burned,
# excluding high-elevation alpine / non-burnable areas that inflate pyrome-mean
# EMC and FM values.  Uses the permissive 'any sub-pixel burned' criterion to
# capture the full fire environment while removing clearly non-fire pixels.

modis_ba = (
    ee.ImageCollection("MODIS/061/MCD64A1")
      .filterDate("2001-01-01", "2023-01-01")
      .filterBounds(co_pyromes.geometry())
      .select("BurnDate")
      .map(lambda img: img.gt(0).unmask(0).byte())  # 1 = burned, 0 = not
)

# Any 500-m pixel burned at least once in the 22-year record
ever_burned_500m = modis_ba.max()  # 0/1 at 500 m

# Aggregate to RTMA 2.5 km: cell is fire-environment if ≥1 sub-pixel burned
# (reduceResolution + reproject is the standard GEE downsampling pattern)
REDUCE_SCALE = 2500  # RTMA native resolution (m); also used in reduce_to_pyromes

fire_mask = (
    ever_burned_500m
      .reduceResolution(reducer=ee.Reducer.max(), maxPixels=64)
      .reproject(crs="EPSG:4326", scale=REDUCE_SCALE)
      .selfMask()   # mask 0s → reduceRegions skips non-fire cells
)
print("Fire-environment mask created from MODIS MCD64A1 (2001–2022)")

### Sanity check — mask coverage per pyrome

Run once manually before submitting exports. All pyromes should have ≥ 20% fire-environment pixel coverage.

In [ ]:
# ── Sanity-check: fire-environment pixel coverage per pyrome ──────────────
# Run this once manually before the export loop to confirm the mask is
# non-trivial for all pyromes.  Expect ≥ 20% coverage in every pyrome.

mask_stats = (
    fire_mask.unmask(0)
      .reduceRegions(
          collection=co_pyromes,
          reducer=ee.Reducer.mean(),   # mean of 0/1 = fraction fire-environment
          scale=REDUCE_SCALE,
          tileScale=4,
      )
      .getInfo()
)
for feat in mask_stats["features"]:
    pid  = feat["properties"].get("PYROME")
    frac = feat["properties"].get("mean", 0) or 0
    flag = " ⚠ < 20%" if frac < 0.20 else ""
    print(f"  Pyrome {pid:>3d}: {frac*100:5.1f}% fire-environment pixels{flag}")

## RTMA ImageCollection — inspect bands

RTMA on GEE exposes `TMP` (2-m temp, **°C**), `DPT` (dew point, **°C**), `SPFH`
(specific humidity, kg/kg), and `ACPC01` (hourly accumulated precip, kg/m² ≡ mm).
Note: the GEE catalog lists TMP and DPT in °C — **not Kelvin** — despite some
third-party examples treating them as K. Confirm with the range check below.

In [ ]:
rtma = ee.ImageCollection('NOAA/NWS/RTMA')
sample_img = rtma.filterDate('2020-07-15', '2020-07-16').first()
print('Bands:', sample_img.bandNames().getInfo())
print('Sample date:', ee.Date(sample_img.get('system:time_start')).format('YYYY-MM-dd HH:mm').getInfo())

# Sanity-check TMP range over CONUS — expect ~-40 to +45 °C, NOT 230–320 K
tmp_stats = sample_img.select('TMP').reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=sample_img.geometry(),
    scale=10000,
    bestEffort=True,
).getInfo()
print('TMP min/max (should be °C, not K):', tmp_stats)

## Per-pixel transforms — RH, VPD, EMC

All computed on RTMA's native grid *before* the pyrome-mean reduction so the
nonlinear NFDRS EMC function isn't biased.

- RH from T, T_d via Magnus saturation vapor pressure ratio
- VPD = e_s(T) − e_s(T_d), in Pa
- EMC: three-regime piecewise NFDRS function (Cohen & Deeming 1985), see
  `fb_tools/weather/nfdrs.py:calc_emc` for the local equivalent.

In [ ]:
TMP_BAND = 'TMP'      # 2-m temperature, °C (NOT Kelvin — GEE RTMA spec)
DPT_BAND = 'DPT'      # 2-m dew point temperature, °C (NOT Kelvin)
PCP_BAND = 'ACPC01'   # 1-hour accumulated precipitation, kg/m² ≡ mm

def add_derived_bands(img):
    """Append tmp_f, rh_pct, vpd_pa, emc_pct, pcp_mm_hr per-pixel bands."""
    # TMP and DPT are already in °C per the RTMA band specification.
    t_c  = img.select(TMP_BAND)
    td_c = img.select(DPT_BAND)

    # Magnus saturation vapor pressure (hPa)
    es_t  = t_c.multiply(17.67).divide(t_c.add(243.5)).exp().multiply(6.112)
    es_td = td_c.multiply(17.67).divide(td_c.add(243.5)).exp().multiply(6.112)
    rh    = es_td.divide(es_t).multiply(100).clamp(0, 100).rename('rh_pct')
    # VPD in Pa (1 hPa = 100 Pa)
    vpd_pa = es_t.subtract(es_td).multiply(100).max(0).rename('vpd_pa')

    tmp_f = t_c.multiply(1.8).add(32).rename('tmp_f')

    # EMC three-regime (Cohen & Deeming 1985): R<10, 10<=R<50, R>=50
    emc_low  = rh.multiply(0.281073).subtract(tmp_f.multiply(rh).multiply(0.000578)).add(0.03229)
    emc_mid  = rh.multiply(0.160107).subtract(tmp_f.multiply(0.014784)).add(2.22749)
    emc_high = (
        rh.pow(2).multiply(0.005565)
          .subtract(tmp_f.multiply(rh).multiply(0.00035))
          .subtract(rh.multiply(0.483199))
          .add(21.0606)
    )
    emc = emc_high.where(rh.lt(50), emc_mid).where(rh.lt(10), emc_low).rename('emc_pct')

    # Server-side conditional — bandNames().contains() avoids a client-side .getInfo()
    # call that would fail inside map().
    pcp = ee.Image(ee.Algorithms.If(
        img.bandNames().contains(PCP_BAND),
        img.select(PCP_BAND).rename('pcp_mm_hr'),
        ee.Image.constant(0).rename('pcp_mm_hr'),
    ))

    return img.addBands([tmp_f, rh, vpd_pa, emc, pcp])

## Per-image pyrome-mean reduction (fire-environment masked)

For each hourly image, apply the MODIS fire-environment mask then call
`reduceRegions` across all 9 CO pyrome geometries in a single batched operation.
Returns one feature per (pyrome, datetime_utc).

**Efficiency vs. prior implementation:**
- `reduceRegions` (one server op) replaces 9 serial `reduceRegion` calls per image.
- Fire mask eliminates alpine/non-burnable pixels (~high-elevation areas that inflate
  pyrome-mean EMC), improving representativeness of the spatial mean.
- `tileScale=4` prevents memory-limit errors on large pyrome geometries.

In [ ]:
REDUCE_BANDS = ['tmp_f', 'rh_pct', 'emc_pct', 'vpd_pa', 'pcp_mm_hr']
# REDUCE_SCALE defined in the fire-mask cell above (2500 m)

def reduce_to_pyromes(img):
    """
    Apply fire-environment mask, compute derived bands, and reduce to
    pyrome-mean scalars in a single batched reduceRegions call.

    Efficiency notes vs. prior implementation:
    - reduceRegions (one server op) replaces 9 serial reduceRegion calls.
    - fire_mask eliminates non-burnable high-elevation pixels, reducing
      pixel count per pyrome and improving representativeness of the mean.
    - tileScale=4 avoids memory-limit errors on large pyrome geometries.
    """
    img_derived = (
        add_derived_bands(img)
          .select(REDUCE_BANDS)
          .updateMask(fire_mask)   # restrict to fire-environment pixels
    )

    # Timestamp computed once, server-side (avoids .getInfo() inside map)
    dt = ee.Date(img.get("system:time_start")).format("YYYY-MM-dd HH:mm:ss")

    return (
        img_derived
          .reduceRegions(
              collection=co_pyromes,
              reducer=ee.Reducer.mean(),
              scale=REDUCE_SCALE,
              tileScale=4,
          )
          .map(lambda f: ee.Feature(None, {
              "pyrome_id":    f.get("PYROME"),
              "datetime_utc": dt,
              "tmp_f":        f.get("tmp_f"),
              "rh_pct":       f.get("rh_pct"),
              "emc_pct":      f.get("emc_pct"),
              "vpd_pa":       f.get("vpd_pa"),
              "pcp_mm_hr":    f.get("pcp_mm_hr"),
          }))
    )

## Quick test — single fire-season day

Sanity-check the reducer on one day before kicking off year-long exports.

In [ ]:
test_day = (
    rtma
    .filterDate('2020-07-15', '2020-07-16')
    .filterBounds(co_pyromes.geometry())
)
print('Hourly images in test day:', test_day.size().getInfo())

test_fc = ee.FeatureCollection(test_day.map(reduce_to_pyromes).flatten())
print('Features (24 hr × 9 pyromes ≈ 216):', test_fc.size().getInfo())

# Pull a small sample for inspection
test_fc.limit(24).getInfo()['features']

## Yearly fire-season exports (2011–2025)

One export task per year. Fire season = April 1 – October 31 (DOY 91–304,
matches GridMET conventions). With `reduceRegions` + fire mask, each task
produces ~7 mo × 30 d × 24 h × 9 pyromes ≈ 45 k rows (same schema as before —
re-exported CSVs are drop-in replacements for existing files).

**Re-export strategy** (fire mask changes require fresh exports for all years):
1. Run the sanity-check cell; confirm ≥ 20% fire-environment coverage per pyrome.
2. Submit one test year (e.g. 2020); download; validate FM in `00b_RTMA-FM.ipynb`.
3. If monthly mean FM100 drops toward 9–11% (vs. current 12–13%), submit remaining years.

Output: Google Drive folder `fb_tools_weather/`, prefix `rtma_hourly_CO_pyromes_YYYY.csv`.

In [ ]:
EXPORT_FOLDER = 'fb_tools_weather'
EXPORT_PREFIX = 'rtma_hourly_CO_pyromes'
YEARS = range(2011, 2026)  # 2011-2025 for complete yearly data

def submit_year(year):
    start = f'{year}-04-01'  # April 1
    end   = f'{year}-11-01'  # November 1 (exclusive)
    season = (
        rtma.filterDate(start, end)
            .filterBounds(co_pyromes.geometry())
            .select(["TMP", "DPT", "ACPC01"])  # drop unused bands early for speed
    )
    fc = ee.FeatureCollection(season.map(reduce_to_pyromes).flatten())

    task = ee.batch.Export.table.toDrive(
        collection=fc,
        description=f'{EXPORT_PREFIX}_{year}',
        folder=EXPORT_FOLDER,
        fileNamePrefix=f'{EXPORT_PREFIX}_{year}',
        fileFormat='CSV',
        selectors=['pyrome_id', 'datetime_utc', 'tmp_f', 'rh_pct', 'emc_pct', 'vpd_pa', 'pcp_mm_hr'],
    )
    task.start()
    return task

# Strategy: run the sanity-check cell first, then test one year before batch.
# 1. Run the mask sanity-check cell above — confirm ≥ 20% coverage per pyrome.
# 2. Uncomment the single-year test below; download CSV; run 00b_RTMA-FM.ipynb
#    on that year alone to verify FM100 seasonal mean drops vs. current exports.
# 3. Once validated, uncomment the full batch submission.

# --- single-year test ---
# tasks = [submit_year(2020)]

# --- full batch (all 15 years) ---
# tasks = [submit_year(y) for y in YEARS]
# for t in tasks: print(t.status())